In [17]:
import sys
from pathlib import Path 
import math
import pickle
import bisect
import numpy as np
import time

In [18]:
ONLINE_JUDGE = True if len(sys.argv) >= 2 and sys.argv[1] == "ONLINE_JUDGE" else False

def debug_print(*args, **kwargs):
    if not ONLINE_JUDGE:
        print(*args, **kwargs)
        
def get_time():
    if ONLINE_JUDGE:
        return time.perf_counter()
    else:
        return time.process_time()

In [19]:
class Env:
    def __init__(self, input_txt_path: Path):
        self.W, self.D, self.N, self.a = self._input(input_txt_path)

    def _input(self, txt_path):
        with open(txt_path, mode="r") as file:
            lines = file.readlines()
        W, D, N = map(int, lines[0].split())
        a = []
        for line, d in zip(lines[1:], range(D)):
            d = list(map(int, line.split()))
            a.append(d)
        return W, D, N, a

In [20]:
W_SIZE = 1000
INF = 10 ** 18
AREA_TH = 0.9975

def fastcopy(obj):
    return pickle.loads(pickle.dumps(obj, -1))

In [21]:
class Area:
    def __init__(self, id_val: int, self_area: int, target_area: int):
        self.id = id_val
        self.self_area = self_area
        self.target_area = target_area
        self.need_first = False

In [22]:
def check_ans(ans: list[tuple[int]]):
    for d in range(len(ans)):
        for coordinates in ans[d]:
            x1, y1, x2, y2 = coordinates
            if x1 < 0 or x2 > W_SIZE or y1 < 0 or y2 > W_SIZE:
                return False
    return True

In [23]:
def get_diff_max(target_areas: list[list[int]], env: Env):
    now_target_areas = fastcopy(target_areas)
    target_dw = len(target_areas)

    get_ind_from_area = [dict() for _ in range(target_dw)]
    for d in range(target_dw):
        tmp_dict = dict()
        for tn in range(env.N):
            val = now_target_areas[d][tn]
            tmp_dict[val] = tn
        get_ind_from_area[d] = tmp_dict

    diff_max = []
    for n in range(env.N):
        tmp_diff_max = []
        len_n = len(now_target_areas[0])
        for d in range(target_dw):
            for tn in range(len_n):
                left_area = now_target_areas[d][tn]
                seach_areas = []
                for next_d in range(target_dw):
                    if d == next_d:
                        seach_areas.append(left_area)
                        continue
                    b_ind = bisect.bisect_left(now_target_areas[next_d], left_area)
                    if b_ind == len_n:
                        break
                    seach_areas.append(now_target_areas[next_d][b_ind])
                if len(seach_areas) != target_dw:
                    continue
                diff = max(seach_areas) - min(seach_areas)
                max_area = max(seach_areas)
                inds = []
                for si, area in enumerate(seach_areas):
                    inds.append(get_ind_from_area[si][area])
                tmp_diff_max.append((diff, max_area, inds, seach_areas))
        sorted_diff_max = sorted(tmp_diff_max)
        add_diff, add_max_area, add_inds, del_areas = sorted_diff_max[0]
        diff_max.append((add_diff, add_max_area, add_inds))
        for si, area in enumerate(del_areas):
            now_target_areas[si].remove(area)

    return diff_max

In [24]:
def calc_cost(ans: list[list[int]], env: Env):
    partial_cost = 0
    area_cost = 0

    hs = set()
    vs = set()
    for d in range(env.D):
        hs2 = set()
        vs2 = set()
        for k in range(env.N):
            i0, j0, i1, j1 = ans[d][k]
            area = (i1 - i0) * (j1 - j0)
            if env.a[d][k] > area:
                area_cost += 100 * (env.a[d][k] - area)
            for j in range(j0, j1):
                if i0 > 0:
                    hs2.add((i0, j))
                if i1 < env.W:
                    hs2.add((i1, j))
            for i in range(i0, i1):
                if j0 > 0:
                    vs2.add((j0, i))
                if j1 < env.W:
                    vs2.add((j1, i))

        if d > 0:
            partial_cost += len(hs ^ hs2)
            partial_cost += len(vs ^ vs2)

        hs = hs2
        vs = vs2
    return partial_cost + area_cost + 1

In [25]:
def dicision_pos(one_day_area: dict[int, int], now_lr, now_ud, x1, y1, x2, y2, how=None):
    area_diffs = []
    for i, area in one_day_area.items():
        len1 = math.ceil(area / now_lr)
        len2 = math.ceil(area / now_ud)
        if len1 * now_lr < len2 * now_ud:
            direct = "ud"
            if how == "diff":
                score = len1 * now_lr - area
            elif how == "aspect":
                score = 1 - min(len1, now_lr) / max(len1, now_lr)
            area_diffs.append((score, len1, direct, i))
        else:
            direct = "lr"
            if how == "diff":
                score = len2 * now_ud - area
            elif how == "aspect":
                score = 1 - min(len2, now_ud) / max(len2, now_ud)
            area_diffs.append((score, len2, direct, i))
    _, min_len, min_direct, min_i = min(area_diffs)
    if min_direct == "ud":
        ans = (min_i, x1, y1, x2, y1 + min_len)
        assigin_area = (x2 - x1) * min_len
        y1 += min_len
    elif min_direct == "lr":
        ans = (min_i, x2 - min_len, y1, x2, y2)
        assigin_area = min_len * (y2 - y1)
        x2 -= min_len
    else:
        raise ValueError("invalid direct")
    
    remain_diff_area = assigin_area - one_day_area[min_i]
    return x1, y1, x2, y2, ans, remain_diff_area

In [26]:
def one_day_greedy_solve(one_day_area: list[int], env: Env):
    x1 = 0
    y1 = 0
    x2 = W_SIZE
    y2 = W_SIZE
    remain_area = W_SIZE * W_SIZE - sum(one_day_area)

    ans = [None for _ in range(env.N)]
    one_day_area_with_index = {}
    for i, area in enumerate(one_day_area):
        one_day_area_with_index[i] = area
    for n in range(env.N):
        remain_n = env.N - n
        now_lr = x2 - x1
        now_ud = y2 - y1
        if n == env.N - 1:
            # 最後の一個は残り全部
            last_ind = list(one_day_area_with_index.keys())[0]
            ans[last_ind] = (x1, y1, x2, y2)
            break

        if remain_n * (W_SIZE - 1) <= remain_area:
            # 残り面積が広いときはアスペクト比を貪欲探索
            x1, y1, x2, y2, now_ans, assigin_area = dicision_pos(one_day_area_with_index, now_lr, now_ud, x1, y1, x2, y2, how="aspect")
        else:
            # 残り面積が狭いときは面積を有効活用
            x1, y1, x2, y2, now_ans, assigin_area = dicision_pos(one_day_area_with_index, now_lr, now_ud, x1, y1, x2, y2, how="diff")
        ans_ind, *ans_tuple = now_ans

        one_day_area_with_index.pop(ans_ind)
                
        ans[ans_ind] = ans_tuple
        remain_area -= assigin_area
    
    return ans

In [27]:
def common_area_solver(target_areas: list[list[int]], env: Env):

    TARGET_DW = len(target_areas)
    diff_max = get_diff_max(target_areas, env)

    # エリアオブジェクトを作成
    area_objs = []
    for target_area in target_areas:
        area_obj = []
        for i in range(env.N):
            area_obj.append(Area(id_val=i, self_area=target_area[i], target_area=target_area[i]))
        area_objs.append(area_obj)

    # 目標面積を決定
    ret_area_objs: list[list[Area]] = fastcopy(area_objs)
    for _, max_area, pattern in diff_max:
        now_area_objs = fastcopy(ret_area_objs)
        for d in range(TARGET_DW):
            now_area_objs[d][pattern[d]].target_area = max_area
            now_area_objs[d][pattern[d]].need_first = True
        is_ok = True
        for d in range(TARGET_DW):
            sum_area = sum(now_area_objs[d][i].target_area for i in range(env.N))
            # if W_SIZE * W_SIZE - sum_area < (W_SIZE - 1) * env.N:
            if sum_area / (W_SIZE * W_SIZE) > AREA_TH:
                is_ok = False
                break
        if is_ok:
            ret_area_objs = fastcopy(now_area_objs)
        else:
            break

    MAX_SUM_AREA = max(sum(x.target_area for x in ret_area_objs[d]) for d in range(TARGET_DW))
    INIT_REMAIN_AREA = W_SIZE * W_SIZE - MAX_SUM_AREA
    ans_all_day = []
    for d in range(TARGET_DW):
        is_need_areas = {}
        is_not_need_areas = {}
        for area_obj in ret_area_objs[d]:
            if area_obj.need_first:
                is_need_areas[area_obj.id] = area_obj.target_area
            else:
                is_not_need_areas[area_obj.id] = area_obj.target_area

        x1 = 0
        y1 = 0
        x2 = W_SIZE
        y2 = W_SIZE
        sum_remain_area = 0
        NOW_SUM_AREA = W_SIZE * W_SIZE - sum(x.target_area for x in ret_area_objs[d])

        ans_one_day = [None for _ in range(env.N)]
        for n in range(env.N):
            remain_n = env.N - n
            now_lr = x2 - x1
            now_ud = y2 - y1
            if is_need_areas:
                if n == env.N - 1:
                    # 最後の一個は残り全部
                    last_ind = list(is_need_areas.keys())[0]
                    ans_one_day[last_ind] = (x1, y1, x2, y2)
                    break
                if remain_n * (W_SIZE - 1) <= (INIT_REMAIN_AREA - sum_remain_area):
                    # 残り面積が広いときはアスペクト比を貪欲探索
                    x1, y1, x2, y2, now_ans, assigin_area = dicision_pos(is_need_areas, now_lr, now_ud, x1, y1, x2, y2, how="aspect")
                else:
                    # 残り面積が狭いときは面積を有効活用
                    x1, y1, x2, y2, now_ans, assigin_area = dicision_pos(is_need_areas, now_lr, now_ud, x1, y1, x2, y2, how="diff")
                ans_ind, *ans_tuple = now_ans

                is_need_areas.pop(ans_ind)
                        
                ans_one_day[ans_ind] = ans_tuple
                sum_remain_area += assigin_area
            elif is_not_need_areas:
                if n == env.N - 1:
                    # 最後の一個は残り全部
                    last_ind = list(is_not_need_areas.keys())[0]
                    ans_one_day[last_ind] = (x1, y1, x2, y2)
                    break
                if remain_n * (W_SIZE - 1) <= (NOW_SUM_AREA - sum_remain_area): # 今日エリアでなくなったら残り面積は独自計算
                    # 残り面積が広いときはアスペクト比を貪欲探索
                    x1, y1, x2, y2, now_ans, assigin_area = dicision_pos(is_not_need_areas, now_lr, now_ud, x1, y1, x2, y2, how="aspect")
                else:
                    # 残り面積が狭いときは面積を有効活用
                    x1, y1, x2, y2, now_ans, assigin_area = dicision_pos(is_not_need_areas, now_lr, now_ud, x1, y1, x2, y2, how="diff")
                ans_ind, *ans_tuple = now_ans

                is_not_need_areas.pop(ans_ind)
                        
                ans_one_day[ans_ind] = ans_tuple
                sum_remain_area += assigin_area
            else:
                raise ValueError("invalid area")
        ans_all_day.append(ans_one_day)

    return ans_all_day

In [28]:
def fin_solver(target_areas: list[list[int]], env: Env):
    if len(target_areas) == 1:
        ret = [one_day_greedy_solve(target_areas[0], env)]
    else:
        ret = common_area_solver(target_areas, env)
        if not check_ans(ret):
            ret = []
            for one_day_area in target_areas:
                ret.append(one_day_greedy_solve(one_day_area, env))
    return ret

In [29]:
def solve(env: Env):
    best_cost = INF
    best_ans = None
    best_shift = None

    start_time = get_time()
    shift_list = list(set([1, 2] + [int(x) for x in np.linspace(3, env.D, 8)]))
    for shift in shift_list:
        ans = []
        for d in range(0, env.D, shift):
            ans += fin_solver(env.a[d:d+shift], env)
        
        if not check_ans(ans):
            raise ValueError("invalid ans")
        cost = calc_cost(ans, env)

        if cost < best_cost:
            best_cost = cost
            best_ans = ans
            best_shift = shift
        if get_time() - start_time > 2.4:
            debug_print("over!!!")
            break

    debug_print(f"best:{best_cost} shift:{best_shift}/{env.D}")
    return best_ans

In [30]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def solve_from_ind(solve_index):
    debug_print(f"----- {solve_index}/100 -----")
    input_txt_path = Path(f"./in/{str(solve_index).zfill(4)}.txt")
    output_txt_path = Path(f"./out/{str(solve_index).zfill(4)}.txt")
    env = Env(input_txt_path)

    ans = solve(env)

    str_ans = []
    for d in range(env.D):
        for k in range(env.N):
            i0, j0, i1, j1 = ans[d][k]
            str_ans.append(f"{i0} {j0} {i1} {j1}")

    with open(output_txt_path, mode="w") as file:
        file.write("\n".join(str_ans))

In [31]:
def main():
    futures = []
    with ThreadPoolExecutor(max_workers=16) as executor:
        for i in range(100):
            future = executor.submit(solve_from_ind, i)
            futures.append(future)

    for future in as_completed(futures):
        future.result()
main()

----- 0/100 -----
----- 1/100 -----
----- 2/100 -----
best:9619 shift:3/5
----- 3/100 -----
----- 4/100 -----
----- 5/100 -----
----- 6/100 -----
----- 7/100 -----
----- 8/100 -----
best:32184 shift:5/5
----- 9/100 ---------- 10/100 -----

----- 11/100 -----
----- 12/100 -----
----- 13/100 -----
----- 14/100 -----
best:10640 shift:6/6
----- 15/100 -----
----- 16/100 -----
best:149034 shift:3/28
----- 17/100 -----
best:169261 shift:3/9
----- 18/100 -----
best:264091 shift:3/25
best:237025 shift:6/26----- 19/100 -----

----- 20/100 -----
----- 21/100 -----
----- 22/100 -----
over!!!
best:726958 shift:2/25
----- 23/100 -----
over!!!
best:416729 shift:3/49
over!!!
best:160619 shift:7/11
----- 24/100 -----
----- 25/100 -----
over!!!
best:105050 shift:9/15
over!!!----- 26/100 -----

best:250745 shift:33/39
over!!!
best:270008 shift:7/18
over!!!
best:414930 shift:5/23
over!!!
best:275190 shift:6/24
----- 27/100 -----
over!!!
best:365281 shift:3/24
----- 28/100 -----
----- 29/100 -----
----- 3

In [32]:
# import time

# start_time = get_time()

# i = 94
# input_txt_path = Path(f"./in/{str(i).zfill(4)}.txt")
# output_txt_path = Path(f"./out/{str(i).zfill(4)}.txt")

# env = Env(input_txt_path)
# ans = solve(env)

# str_ans = []
# for d in range(env.D):
#     for k in range(env.N):
#         i0, j0, i1, j1 = ans[d][k]
#         str_ans.append(f"{i0} {j0} {i1} {j1}")

# with open(output_txt_path, mode="w") as file:
#     file.write("\n".join(str_ans))

# debug_print(get_time() - start_time)